In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga del shapefile de secciones censales

In [2]:
# Cargo el GDF con las secciones censales
ruta_shp_secciones = os.path.join(DATA_OUTPUTS_DIR, "Shapefiles", "cyl_2022.shp")
gdf_secciones = gpd.read_file(ruta_shp_secciones)

# Compruebo que se haya cargado
print(f"Vista del gdf de secciones:\n{gdf_secciones.head()}")

Vista del gdf de secciones:
        CUSEC  CUMUN CSEC CDIS CMUN CPRO CCA    CUDIS  CLAU2   NPRO  \
0  0500101001  05001  001   01  001   05  07  0500101  05001  Ávila   
1  0500201001  05002  001   01  002   05  07  0500201  05002  Ávila   
2  0500201002  05002  002   01  002   05  07  0500201  05002  Ávila   
3  0500501001  05005  001   01  005   05  07  0500501  05005  Ávila   
4  0500701001  05007  001   01  007   05  07  0500701  05007  Ávila   

               NCA CNUT0 CNUT1 CNUT2 CNUT3                      NMUN  \
0  Castilla y León    ES     4     1     1                   Adanero   
1  Castilla y León    ES     4     1     1                Adrada, La   
2  Castilla y León    ES     4     1     1                Adrada, La   
3  Castilla y León    ES     4     1     1                  Albornos   
4  Castilla y León    ES     4     1     1  Aldeanueva de Santa Cruz   

                                            geometry  
0  POLYGON ((365705.918 4536187.034, 365958.915 4...  
1 

# Carga del fichero de centros de salud

El fichero de centros de salud en formato shapefile se extrae de datosabiertos.jcyl disponible en la siguiente dirección: 

https://idecyl.jcyl.es/geonetwork/srv/spa/catalog.search#/metadata/SPAGOBCYLCITDTSHHCES

In [3]:
# Cargo el fichero csv con los centros docentes
ruta_shp_centros_salud = os.path.join(DATA_INPUTS_DA, "salud_cyl_centros", "salud_cyl_centros.shp")
gdf_centros_salud = gpd.read_file(ruta_shp_centros_salud).to_crs(epsg=4326)
gdf_centros_salud["centro_salud_id"] = range(1, len(gdf_centros_salud) + 1)
# Veo una muestra de su estructura y contenido
print(gdf_centros_salud.info())
gdf_centros_salud.sample(5)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 490 entries, 0 to 489
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   fid              490 non-null    int32   
 1   d_tipocent       490 non-null    object  
 2   a_nombre         490 non-null    object  
 3   a_complejo       20 non-null     object  
 4   a_direccio       490 non-null    object  
 5   n_latitud        490 non-null    object  
 6   n_longitud       490 non-null    object  
 7   a_cod_post       490 non-null    object  
 8   geometry         488 non-null    geometry
 9   centro_salud_id  490 non-null    int64   
dtypes: geometry(1), int32(1), int64(1), object(7)
memory usage: 36.5+ KB
None


,fid,d_tipocent,a_nombre,a_complejo,a_direccio,n_latitud,n_longitud,a_cod_post,geometry,centro_salud_id
208,209,Centro de Salud,Ávila Sur Oeste,None,"C/ FRAY GIL, 10",40.6483543,-4.6961111,05003,POINT (-4.69611 40.64835),209
83,84,Centro de Salud,Arturo Eyries,None,"C/ PUERTO RICO, 1",41.6343117,-4.748222,47014,POINT (-4.74822 41.63431),84
381,383,Punto de Atención Continuada,Burgohondo,None,"PZA. MAYOR, S/N",40.4145144,-4.7855486,05113,None,382
456,458,Punto de Atención Continuada,Tamames,None,"PZA. FERIAL, S/N",40.6562382,-6.1077892,37600,POINT (-6.10779 40.65624),457
245,246,Centro de Salud,Villada,None,"C/ CARLOS CASADO DEL ALISAL, S/N",42.2488777,-4.9661323,34340,POINT (-4.96613 42.24888),246


# Filtrado del fichero

## Tipo de centro
Como se dispone de varios tipos de centro, se van a agrupar en 2 grandes grupos, centros_primaria y centros_especializada según:

In [4]:
# Vemos el numero de centros por tipo
print(f"Tipos de centro:\n{gdf_centros_salud['d_tipocent'].value_counts()}")

# Separo centros de primaria de especializada
group_map = {
    "Hospital": "especializada",
    "Centro de especialidades o polivalente": "especializada",
    "Centro de Salud": "primaria",
    "Punto de Atención Continuada": "primaria",
    "Centro de Guardia": "primaria",
}

gdf_centros_salud["Grupo"] = gdf_centros_salud["d_tipocent"].map(group_map)

print(f"Grupos de centro:\n{gdf_centros_salud['Grupo'].value_counts()}")


Tipos de centro:
d_tipocent
Centro de Salud                           247
Punto de Atención Continuada              179
Hospital                                   26
Centro de Guardia                          20
Centro de especialidades o polivalente     18
Name: count, dtype: int64
Grupos de centro:
Grupo
primaria         446
especializada     44
Name: count, dtype: int64


# Asignación de código de sección censal (CUSEC)
Como no se dispone por defecto del CUSEC pero sí de las coordenadas geográficas del centro, y se dispone de un shapefile con los
códigos de sección de Castilla y León georreferenciados, puede asignarse el CUSEC a este conjunto de datos mediante un join espacial

In [5]:
# Las columnas de latitud y longitud del gdf están con errores de codificacion. Las extraigo nuevamente de la geometria
gdf_centros_salud["n_longitud"] = gdf_centros_salud.geometry.x
gdf_centros_salud["n_latitud"] = gdf_centros_salud.geometry.y

# Utilizo una función definida para integrar el cusec

gdf_centros_salud_final = asignar_cusec_por_coordenadas(
    gdf_centros_salud, "n_longitud", "n_latitud")

# Compruebo que tiene ahora la columna cusec
print(f"{gdf_centros_salud_final.sample(5)}\n")

# Miro a ver si hay alguno sin CUSEC
print(f"Centros sin CUSEC asignado: {gdf_centros_salud_final['CUSEC'].isna().sum()}")

# Hay 2 sin cusec así que asigno por CP
gdf_centros_salud_final = asignar_cusec_por_cp(gdf_centros_salud_final, columna_cp = "a_cod_post")

     fid                              d_tipocent  \
478  481                       Centro de Guardia   
451  453            Punto de Atención Continuada   
68    69                         Centro de Salud   
307  308            Punto de Atención Continuada   
278  279  Centro de especialidades o polivalente   

                                   a_nombre a_complejo  \
478                   Quintanilla de Losada       None   
451                        Fuentes de Oñoro       None   
68                    Carrión de los Condes       None   
307                              Villadiego       None   
278  Centro de Especialidades Arturo Eyries       None   

                   a_direccio  n_latitud  n_longitud a_cod_post  \
478                C/ REAL, 1  42.278579   -6.560626      24717   
451       C/ PEDRO MATEOS, 58  40.597326   -6.820729      37480   
68   PZA. CONDE DE GARAY, S/N  42.336064   -4.601673      34120   
307   C/ REYES CATÓLICOS, S/N  42.514207   -4.010227      09120   
278

# Cálculo de distancia mínima a un centro desde un CUSEC y accesibilidad de centros para un CUSEC
Es necesario para cada sección censal calcular la distancia mínima existente hacia un centro de un tipo, así como crear
una medida de accesibilidad de un centro concreto desde una sección cens

In [7]:
# Recorremos cada categoría educativa
df_resultado = pd.DataFrame({"CUSEC": gdf_secciones["CUSEC"].unique()})

# Tabla global de relaciones
lista_relaciones = []

for grupo in gdf_centros_salud_final["Grupo"].dropna().unique():
    print(f"\n➡️ Procesando grupo: {grupo}")

    # Filtrar centros de la categoría
    df_grupo = gdf_centros_salud_final[gdf_centros_salud_final["Grupo"] == grupo].copy()

    # Calcular accesibilidad (versión v2 o v3)
    df_acc, df_rel = calcular_accesibilidad_v2(
        df_servicio=df_grupo,
        nombre_servicio=grupo,
        col_id="centro_salud_id"
    )

    # Acumular relaciones
    lista_relaciones.append(df_rel)

    # Unir al DF principal
    df_resultado = df_resultado.merge(df_acc, on="CUSEC", how="outer")

df_relaciones_global = pd.concat(lista_relaciones, ignore_index=True)

df_resultado.head()


➡️ Procesando grupo: especializada


Accesibilidad especializada: 100%|███████████████████████████████████████████| 3535/3535 [00:06<00:00, 574.77sección/s]



➡️ Procesando grupo: primaria


Accesibilidad primaria: 100%|████████████████████████████████████████████████| 3535/3535 [00:10<00:00, 324.94sección/s]


,CUSEC,dist_min_especializada_km,n_especializada_1km,n_especializada_5km,n_especializada_15km,n_especializada_30km,disp_ponderada_especializada,dist_min_primaria_km,n_primaria_1km,n_primaria_5km,n_primaria_15km,n_primaria_30km,disp_ponderada_primaria
0,0500101001,27.565760,0,0,0,1,0.1,12.416063,0,0,2,10,1.4
1,0500201001,36.402927,0,0,0,0,0.0,6.209785,0,0,2,9,1.3
2,0500201002,34.804173,0,0,0,0,0.0,4.415298,0,2,5,9,2.5
3,0500501001,24.920614,0,0,0,3,0.3,4.689838,0,2,4,19,3.3
4,0500701001,29.082239,0,0,0,1,0.1,7.357844,0,0,4,8,1.6


# Export de los csv construidos

In [9]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DA, exist_ok=True)

# Rutas de salida
ruta_resultado = os.path.join(DATA_OUTPUTS_DA, "accesibilidad_centros_salud.csv")
ruta_centros = os.path.join(DATA_OUTPUTS_DA, "centros_salud_final.csv")
ruta_relaciones = os.path.join(DATA_OUTPUTS_DA, "relaciones_centros_salud.csv")

# Guardar DataFrames
df_resultado.to_csv(ruta_resultado, index=False, encoding="utf-8-sig")
gdf_centros_salud_final.to_csv(ruta_centros, index=False, encoding="utf-8-sig")

df_relaciones_global.to_csv(
    ruta_relaciones,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

print(f"✅ Archivos guardados correctamente en:\n- {DATA_OUTPUTS_DA}")

✅ Archivos guardados correctamente en:
- D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DA_Dim_servicios
